# Lesson 19 — 并发任务：采样、发送和按钮不能互相拖住

网络发送可能暂时离线，但传感器采样和按钮响应仍应继续。先用确定性的 host `tick()` 模型观察 producer/consumer，再用 `%concurrency_probe` 运行真实 `std::thread`。这不是 FreeRTOS scheduler；ESP-IDF 的 Task 栈、优先级、ISR 和 watchdog 要到 STEP 2 在真实环境验证。

**学习顺序：** 先运行 guided，再修改，故意运行 broken，最后自己完成 exercise。答案 Notebook 独立保存。

In [ ]:
from pathlib import Path
project = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tools/notebook_setup.py").is_file())
exec((project / "tools/notebook_setup.py").read_text())

## 1. Guided

先观察面板输出，再读 C++。

In [ ]:
%%academy_lab 19-concurrent-tasks guided guided pass
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& d) : device_(d) {
        }
        void tick();

    private:
        Device& device_;
        std::uint32_t next_sensor_ms_ = 0, next_network_ms_ = 0;
        bool previous_button_ = false;
        int pending_samples_ = 0;
    };
    void Controller::tick() {
        auto now = device_.input.now_ms;
        if (now >= next_sensor_ms_) {
            device_.output.reading = device_.input.temperature;
            ++pending_samples_;
            next_sensor_ms_ = now + 100;
        }
        if (device_.input.connected && pending_samples_ > 0 &&
            now >= next_network_ms_) {
            --pending_samples_;
            ++device_.output.events;
            device_.output.outgoing = "temperature sample";
            next_network_ms_ = now + 50;
        }
        if (device_.input.button && !previous_button_)
            device_.output.led = !device_.output.led;
        previous_button_ = device_.input.button;
        device_.output.queue_depth = pending_samples_;
        device_.output.state = device_.input.connected ? "online" : "offline";
    }
}

## 2. Modify

把 `next_sensor_ms_=now+100` 改成 `now+200`，再观察 queue_depth 和发送频率的变化。

In [ ]:
%%academy_lab 19-concurrent-tasks modify
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& d) : device_(d) {
        }
        void tick();

    private:
        Device& device_;
        std::uint32_t next_sensor_ms_ = 0, next_network_ms_ = 0;
        bool previous_button_ = false;
        int pending_samples_ = 0;
    };
    void Controller::tick() {
        auto now = device_.input.now_ms;
        if (now >= next_sensor_ms_) {
            device_.output.reading = device_.input.temperature;
            ++pending_samples_;
            next_sensor_ms_ = now + 100;
        }
        if (device_.input.connected && pending_samples_ > 0 &&
            now >= next_network_ms_) {
            --pending_samples_;
            ++device_.output.events;
            device_.output.outgoing = "temperature sample";
            next_network_ms_ = now + 50;
        }
        if (device_.input.button && !previous_button_)
            device_.output.led = !device_.output.led;
        previous_button_ = device_.input.button;
        device_.output.queue_depth = pending_samples_;
        device_.output.state = device_.input.connected ? "online" : "offline";
    }
}

## 3. Broken

下面的检查应该失败；先观察故障证据，再查看分层提示。

In [ ]:
%%academy_lab 19-concurrent-tasks broken exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& d) : device_(d) {
        }
        void tick() {
            auto now = device_.input.now_ms;
            if (device_.input.connected && now >= next_) {
                device_.output.reading = device_.input.temperature;
                ++pending_;
                next_ = now + 100;
            }
            if (device_.input.connected && pending_ > 0) {
                --pending_;
                ++device_.output.events;
            }
            device_.output.queue_depth = pending_;
        }

    private:
        Device& device_;
        std::uint32_t next_ = 0;
        int pending_ = 0;
    };
}

## 4. Build

要求：离线时每 100 ms 采样，恢复连接后每次 tick 最多发送一个样本；按钮新按下时立即切换 LED。

提示 1：三个动作有三个不同的时间条件。提示 2：离线只应阻止发送。提示 3：`pending_samples_` 是 producer/consumer 队列的最小 host 模型。

In [ ]:
%%academy_lab 19-concurrent-tasks exercise exercise
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& d) : device_(d) {
        }
        void tick() { /* TODO: 你的实现 */
        }

    private:
        Device& device_;
    };
}


## 5. 真实 host 线程探针

运行下一格；它真的启动有限的 std::thread，然后 join。它不是 FreeRTOS。

In [ ]:
%concurrency_probe

## 6. 反思

写下：这个模型在 host 上证明了什么？哪些行为必须买板后重新验证？